# SurfaceTarget-CRC — phase 03 — patch 3 1

Extracted from the original Google Colab research notebook.

**Note:** Earlier patches and project data may be required to execute these cells.


In [ ]:
# ============================================================
# SURFACETARGET-CRC
# PHASE 3 — PATCH 3.1
#
# HUMAN PROTEIN ATLAS ACQUISITION AND ANNOTATION AUDIT
#
# Objectives:
# 1. Load validated transcriptomic/proteomic evidence
# 2. Download official HPA subcellular annotations
# 3. Download official HPA normal-tissue IHC data
# 4. Preserve source files and SHA256 checksums
# 5. Match annotations using stable Ensembl gene IDs
# 6. Identify reported plasma-membrane localization
# 7. Summarize normal-tissue IHC coverage
# 8. Review leading genes
# 9. Save annotations and provenance
#
# IMPORTANT:
# - No final target ranking
# - No automatic candidate exclusions
# - No assumption of extracellular accessibility
# - No assumption that missing HPA data means absence
# - No changes to TCGA or CPTAC analyses
# ============================================================

from pathlib import Path
from datetime import datetime, timezone

import pandas as pd
import numpy as np

import requests
import hashlib
import json
import time
import os

from zipfile import ZipFile, is_zipfile


# ------------------------------------------------------------
# 1. PROJECT CONFIGURATION
# ------------------------------------------------------------

ROOT = Path("/content/SurfaceTarget-CRC")

META = ROOT / "data/metadata"

RAW = ROOT / "data/raw/hpa"

RESULTS = ROOT / "results/tables"

for folder in [META, RAW, RESULTS]:
    folder.mkdir(parents=True, exist_ok=True)

INPUT_PATH = (
    RESULTS /
    "cptac_mss_extended_transcriptomic_integration.csv"
)

print("=" * 65)
print("SURFACETARGET-CRC")
print("PHASE 3 — PATCH 3.1")
print("HUMAN PROTEIN ATLAS ANNOTATION AUDIT")
print("=" * 65)


# ------------------------------------------------------------
# 2. LOAD EXISTING CROSS-OMICS CANDIDATES
# ------------------------------------------------------------

if not INPUT_PATH.exists():
    raise FileNotFoundError(
        "Patch 2.4 integrated candidate table missing."
    )

candidates = pd.read_csv(
    INPUT_PATH,
    dtype={"gene_id": str},
    low_memory=False
)

required_columns = {
    "gene_id",
    "gene_name",
    "protein_enriched",
    "combined_transcriptomic_support"
}

if not required_columns.issubset(candidates.columns):
    raise RuntimeError(
        "Required candidate columns are missing."
    )

if len(candidates) != 948:
    raise RuntimeError(
        "Unexpected transcriptomic candidate count."
    )

if candidates["gene_id"].duplicated().any():
    raise RuntimeError(
        "Duplicate Ensembl gene IDs in candidate table."
    )

# TCGA Ensembl IDs include version suffixes.
#
# Example:
# ENSG00000062038.15 -> ENSG00000062038
#
# We retain original IDs while creating a
# separate stable identifier for HPA matching.

candidates["ensembl_stable_id"] = (
    candidates["gene_id"]
    .astype(str)
    .str.replace(
        r"\.\d+$",
        "",
        regex=True
    )
)

valid_ensembl = (
    candidates["ensembl_stable_id"]
    .str.fullmatch(r"ENSG\d{11}")
)

if not valid_ensembl.all():
    raise RuntimeError(
        "Unexpected Ensembl gene identifier format."
    )

if candidates["ensembl_stable_id"].duplicated().any():
    raise RuntimeError(
        "Duplicate stable Ensembl identifiers."
    )

candidates["protein_enriched"] = (
    candidates["protein_enriched"]
    .fillna(False)
    .astype(str)
    .str.lower()
    .eq("true")
)

print("\nCandidate table loaded.")

print("Transcriptomic candidates:", len(candidates))

print(
    "Cross-omics-supported genes:",
    int(candidates["protein_enriched"].sum())
)

if int(candidates["protein_enriched"].sum()) != 20:
    raise RuntimeError(
        "Expected 20 cross-omics-supported genes."
    )

print("\nCandidate identifier validation: PASSED")


# ------------------------------------------------------------
# 3. DEFINE OFFICIAL HPA DOWNLOADS
# ------------------------------------------------------------

HPA_SOURCES = {

    "subcellular": {
        "url": (
            "https://www.proteinatlas.org/download/"
            "tsv/subcellular_location.tsv.zip"
        ),
        "filename": "subcellular_location.tsv.zip"
    },

    "normal_ihc": {
        "url": (
            "https://www.proteinatlas.org/download/"
            "tsv/normal_ihc_data.tsv.zip"
        ),
        "filename": "normal_ihc_data.tsv.zip"
    }
}

print("\nOfficial HPA datasets requested:", len(HPA_SOURCES))


# ------------------------------------------------------------
# 4. DOWNLOAD AND INTEGRITY FUNCTIONS
# ------------------------------------------------------------

session = requests.Session()

session.headers.update({
    "User-Agent": "SurfaceTarget-CRC-Research/1.0",
    "Accept": "application/zip,*/*"
})


def sha256_file(path):

    digest = hashlib.sha256()

    with open(path, "rb") as handle:

        for chunk in iter(
            lambda: handle.read(1024 * 1024),
            b""
        ):
            digest.update(chunk)

    return digest.hexdigest()


def inspect_zip(path):

    if not is_zipfile(path):
        raise RuntimeError(
            f"File is not a valid ZIP archive: {path}"
        )

    with ZipFile(path) as archive:

        members = [
            info
            for info in archive.infolist()
            if not info.is_dir()
            and info.filename.lower().endswith(".tsv")
        ]

        if len(members) != 1:
            raise RuntimeError(
                "Expected one TSV file per HPA archive. "
                f"Found: {[m.filename for m in members]}"
            )

        member = members[0]

        if member.file_size > 500 * 1024**2:
            raise RuntimeError(
                "Unexpectedly large uncompressed HPA file."
            )

        with archive.open(member) as handle:
            header = (
                handle.readline()
                .decode("utf-8-sig")
                .rstrip("\r\n")
                .split("\t")
            )

    return member.filename, header


def download_hpa(url, destination):

    if destination.exists():

        try:
            inspect_zip(destination)

            print("Existing valid archive reused.")

            return "Existing"

        except Exception:
            print("Existing archive invalid; downloading again.")

    temporary = Path(str(destination) + ".part")

    for attempt in range(1, 4):

        try:

            if temporary.exists():
                temporary.unlink()

            with session.get(
                url,
                stream=True,
                timeout=(30, 120)
            ) as response:

                response.raise_for_status()

                total_bytes = 0

                with open(temporary, "wb") as output:

                    for chunk in response.iter_content(
                        chunk_size=1024 * 1024
                    ):

                        if not chunk:
                            continue

                        output.write(chunk)

                        total_bytes += len(chunk)

                        if total_bytes > 150 * 1024**2:
                            raise RuntimeError(
                                "Download exceeded the safety limit."
                            )

            inspect_zip(temporary)

            os.replace(temporary, destination)

            return "Downloaded"

        except Exception as error:

            print(
                f"Download attempt {attempt}/3 failed: {error}"
            )

            if temporary.exists():
                temporary.unlink()

            if attempt == 3:
                raise

            time.sleep(2 ** attempt)


# ------------------------------------------------------------
# 5. ACQUIRE HPA DATASETS
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("DOWNLOADING HUMAN PROTEIN ATLAS DATA")
print("-" * 65)

source_records = []

source_paths = {}

for name, source in HPA_SOURCES.items():

    destination = RAW / source["filename"]

    print("\nDataset:", name)

    status = download_hpa(
        source["url"],
        destination
    )

    archive_member, header = inspect_zip(
        destination
    )

    source_paths[name] = destination

    source_records.append({

        "dataset": name,
        "source_url": source["url"],
        "local_path": str(destination),
        "archive_member": archive_member,
        "download_status": status,
        "size_bytes": destination.stat().st_size,
        "sha256": sha256_file(destination),
        "columns": header

    })

    print("Status:", status)

    print(
        "Compressed size (MiB):",
        round(destination.stat().st_size / 1024**2, 2)
    )

    print("Archive member:", archive_member)

    print("Columns:", header)

print("\nHPA source acquisition: COMPLETE")


# ------------------------------------------------------------
# 6. LOAD SUBCELLULAR LOCATION DATA
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("LOADING HPA SUBCELLULAR ANNOTATIONS")
print("-" * 65)

sub_path = source_paths["subcellular"]

with ZipFile(sub_path) as archive:

    member = next(
        info.filename
        for info in archive.infolist()
        if info.filename.lower().endswith(".tsv")
    )

    with archive.open(member) as handle:

        subcellular = pd.read_csv(
            handle,
            sep="\t",
            dtype=str,
            keep_default_na=False
        )

subcellular.columns = (
    subcellular.columns.str.strip()
)

required_sub_columns = {
    "Gene",
    "Gene name",
    "Reliability",
    "Enhanced",
    "Supported",
    "Approved",
    "Uncertain"
}

if not required_sub_columns.issubset(
    subcellular.columns
):
    raise RuntimeError(
        "Unexpected HPA subcellular data schema. "
        f"Missing: {required_sub_columns - set(subcellular.columns)}"
    )

subcellular["Gene"] = (
    subcellular["Gene"]
    .astype(str)
    .str.strip()
    .str.replace(r"\.\d+$", "", regex=True)
)

if subcellular["Gene"].duplicated().any():
    raise RuntimeError(
        "Duplicate HPA subcellular Ensembl IDs."
    )

print("\nTotal HPA subcellular records:", len(subcellular))


# ------------------------------------------------------------
# 7. SELECT CANDIDATE SUBCELLULAR ANNOTATIONS
# ------------------------------------------------------------

candidate_gene_ids = set(
    candidates["ensembl_stable_id"]
)

sub_candidate = subcellular[
    subcellular["Gene"].isin(candidate_gene_ids)
].copy()

sub_candidate = sub_candidate.rename(
    columns={
        "Gene": "ensembl_stable_id",
        "Gene name": "hpa_gene_name",
        "Reliability": "hpa_subcellular_reliability",
        "Enhanced": "hpa_enhanced",
        "Supported": "hpa_supported",
        "Approved": "hpa_approved",
        "Uncertain": "hpa_uncertain"
    }
)

sub_columns = [
    "ensembl_stable_id",
    "hpa_gene_name",
    "hpa_subcellular_reliability",
    "hpa_enhanced",
    "hpa_supported",
    "hpa_approved",
    "hpa_uncertain"
]

sub_candidate = sub_candidate[
    sub_columns
].copy()

print(
    "Candidates with HPA subcellular records:",
    len(sub_candidate)
)


# ------------------------------------------------------------
# 8. IDENTIFY REPORTED PLASMA-MEMBRANE LOCATION
# ------------------------------------------------------------

# This identifies explicitly reported HPA
# plasma-membrane localization.
#
# It does NOT establish:
# - Extracellular epitope accessibility
# - Antibody binding
# - Tumor-cell-specific localization
# - Therapeutic suitability

confident_columns = [
    "hpa_enhanced",
    "hpa_supported",
    "hpa_approved"
]


def combine_locations(row, columns):

    values = [
        str(row[col]).strip()
        for col in columns
        if str(row[col]).strip()
    ]

    return "; ".join(values)


sub_candidate["hpa_reported_locations"] = (
    sub_candidate.apply(
        lambda row: combine_locations(
            row,
            confident_columns
        ),
        axis=1
    )
)

sub_candidate["hpa_plasma_membrane_reported"] = (
    sub_candidate["hpa_reported_locations"]
    .str.contains(
        r"\bplasma membrane\b",
        case=False,
        regex=True,
        na=False
    )
)

sub_candidate["hpa_cell_junctions_reported"] = (
    sub_candidate["hpa_reported_locations"]
    .str.contains(
        r"\bcell junctions?\b",
        case=False,
        regex=True,
        na=False
    )
)

sub_candidate["hpa_uncertain_plasma_membrane"] = (
    sub_candidate["hpa_uncertain"]
    .str.contains(
        r"\bplasma membrane\b",
        case=False,
        regex=True,
        na=False
    )
)

print("\nHPA subcellular annotation summary:")

print(
    "Plasma membrane reported:",
    int(
        sub_candidate[
            "hpa_plasma_membrane_reported"
        ].sum()
    )
)

print(
    "Cell junctions reported:",
    int(
        sub_candidate[
            "hpa_cell_junctions_reported"
        ].sum()
    )
)

print(
    "Uncertain plasma-membrane annotation:",
    int(
        sub_candidate[
            "hpa_uncertain_plasma_membrane"
        ].sum()
    )
)


# ------------------------------------------------------------
# 9. LOAD NORMAL-TISSUE IHC RECORDS
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("LOADING HPA NORMAL-TISSUE IHC")
print("-" * 65)

# The full HPA normal-tissue table can be large.
#
# Read in chunks and retain only records
# corresponding to our 948 candidate genes.
#
# All original downloaded files are preserved.

ihc_path = source_paths["normal_ihc"]

ihc_chunks = []

total_ihc_records = 0

with ZipFile(ihc_path) as archive:

    member = next(
        info.filename
        for info in archive.infolist()
        if info.filename.lower().endswith(".tsv")
    )

    with archive.open(member) as handle:

        reader = pd.read_csv(
            handle,
            sep="\t",
            dtype=str,
            keep_default_na=False,
            chunksize=75000,
            low_memory=False
        )

        for chunk in reader:

            chunk.columns = (
                chunk.columns.str.strip()
            )

            required_ihc_columns = {
                "Gene",
                "Tissue",
                "Cell type",
                "Level",
                "Reliability"
            }

            if not required_ihc_columns.issubset(
                chunk.columns
            ):
                raise RuntimeError(
                    "Unexpected normal-tissue IHC schema. "
                    f"Missing: {required_ihc_columns - set(chunk.columns)}"
                )

            total_ihc_records += len(chunk)

            chunk["Gene"] = (
                chunk["Gene"]
                .astype(str)
                .str.strip()
                .str.replace(
                    r"\.\d+$",
                    "",
                    regex=True
                )
            )

            matched = chunk[
                chunk["Gene"].isin(candidate_gene_ids)
            ].copy()

            if not matched.empty:
                ihc_chunks.append(matched)

print("\nTotal HPA normal IHC records scanned:",
      total_ihc_records)

if not ihc_chunks:
    raise RuntimeError(
        "No normal-tissue IHC records matched candidates."
    )

ihc_candidate = pd.concat(
    ihc_chunks,
    ignore_index=True
)

print(
    "Candidate normal-tissue IHC records:",
    len(ihc_candidate)
)

print(
    "Candidates with normal-tissue IHC records:",
    ihc_candidate["Gene"].nunique()
)


# ------------------------------------------------------------
# 10. NORMAL-TISSUE EXPRESSION COVERAGE
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("NORMAL-TISSUE EXPRESSION SUMMARY")
print("-" * 65)

ihc_candidate["level_normalized"] = (
    ihc_candidate["Level"]
    .astype(str)
    .str.strip()
    .str.lower()
)

print("\nObserved IHC expression labels:")

print(
    ihc_candidate[
        "level_normalized"
    ].value_counts().to_string()
)

# These are descriptive HPA IHC categories.
#
# We do not convert categorical staining
# levels into absolute protein concentrations.
#
# "High" and "Medium" are reported
# expression categories, not safety outcomes.

ihc_candidate["is_high"] = (
    ihc_candidate["level_normalized"]
    .eq("high")
)

ihc_candidate["is_medium_or_high"] = (
    ihc_candidate["level_normalized"]
    .isin(["medium", "high"])
)


summary_records = []

for gene_id, group in ihc_candidate.groupby(
    "Gene",
    sort=False
):

    high = group[
        group["is_high"]
    ]

    medium_high = group[
        group["is_medium_or_high"]
    ]

    summary_records.append({

        "ensembl_stable_id": gene_id,

        "hpa_ihc_records": len(group),

        "hpa_normal_tissues_profiled":
            group["Tissue"].nunique(),

        "hpa_normal_tissues_high":
            high["Tissue"].nunique(),

        "hpa_normal_tissues_medium_high":
            medium_high["Tissue"].nunique(),

        "hpa_high_tissue_names":
            "; ".join(
                sorted(
                    high["Tissue"]
                    .dropna()
                    .unique()
                )
            ),

        "hpa_medium_high_tissue_names":
            "; ".join(
                sorted(
                    medium_high["Tissue"]
                    .dropna()
                    .unique()
                )
            )
    })


ihc_summary = pd.DataFrame(
    summary_records
)

if ihc_summary["ensembl_stable_id"].duplicated().any():
    raise RuntimeError(
        "Duplicate gene-level IHC summaries."
    )

print("\nIHC summaries calculated:",
      len(ihc_summary))


# ------------------------------------------------------------
# 11. INTEGRATE HPA ANNOTATIONS
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("INTEGRATING HPA EVIDENCE")
print("-" * 65)

integrated = candidates.merge(
    sub_candidate,
    on="ensembl_stable_id",
    how="left",
    validate="one_to_one"
)

integrated = integrated.merge(
    ihc_summary,
    on="ensembl_stable_id",
    how="left",
    validate="one_to_one"
)

if len(integrated) != 948:
    raise RuntimeError(
        "HPA integration changed candidate count."
    )

integrated["hpa_subcellular_record_available"] = (
    integrated["hpa_gene_name"].notna()
)

integrated["hpa_normal_ihc_record_available"] = (
    integrated["hpa_ihc_records"].notna()
)

print("\nIntegration: PASSED")

print("Candidate genes preserved:", len(integrated))

print(
    "With HPA subcellular records:",
    int(
        integrated[
            "hpa_subcellular_record_available"
        ].sum()
    )
)

print(
    "With normal-tissue IHC records:",
    int(
        integrated[
            "hpa_normal_ihc_record_available"
        ].sum()
    )
)

print(
    "With reported plasma-membrane localization:",
    int(
        integrated[
            "hpa_plasma_membrane_reported"
        ].fillna(False).sum()
    )
)

print(
    "Cross-omics-supported genes:",
    int(integrated["protein_enriched"].sum())
)


# ------------------------------------------------------------
# 12. REVIEW LEADING GENES
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("LEADING GENE HPA ANNOTATIONS")
print("-" * 65)

leading_genes = [
    "CDH3",
    "CLDN1",
    "FAP",
    "DPEP1",
    "SLC7A5",
    "SLC5A6",
    "CTHRC1",
    "S100P",
    "KRT80",
    "CEMIP",
    "INHBA",
    "WNT2"
]

leading = integrated[
    integrated["gene_name"].isin(leading_genes)
].copy()

leading["display_order"] = (
    leading["gene_name"].map({
        gene: index
        for index, gene in enumerate(leading_genes)
    })
)

leading = leading.sort_values(
    "display_order"
)

display_columns = [
    "gene_name",
    "protein_enriched",
    "hpa_subcellular_record_available",
    "hpa_reported_locations",
    "hpa_plasma_membrane_reported",
    "hpa_normal_tissues_profiled",
    "hpa_normal_tissues_medium_high"
]

print("\n")

print(
    leading[display_columns]
    .to_string(index=False)
)

print("\nIMPORTANT:")

print(
    "HPA subcellular localization is not "
    "equivalent to extracellular accessibility."
)

print(
    "Normal-tissue IHC expression is not "
    "equivalent to demonstrated clinical toxicity."
)

print(
    "Missing HPA annotation is not "
    "evidence of biological absence."
)


# ------------------------------------------------------------
# 13. SAVE HPA AUDIT OUTPUTS
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("SAVING HPA ANNOTATION TABLES")
print("-" * 65)

pd.DataFrame(source_records).to_csv(
    META / "hpa_source_manifest.csv",
    index=False
)

sub_candidate.to_csv(
    RESULTS / "hpa_candidate_subcellular_annotations.csv",
    index=False
)

ihc_candidate.to_csv(
    RESULTS / "hpa_candidate_normal_ihc_records.csv",
    index=False
)

ihc_summary.to_csv(
    RESULTS / "hpa_candidate_normal_ihc_summary.csv",
    index=False
)

integrated.to_csv(
    RESULTS / "surfacetarget_hpa_annotation_audit.csv",
    index=False
)

leading.to_csv(
    RESULTS / "surfacetarget_hpa_leading_gene_review.csv",
    index=False
)

print("\nAll HPA annotation tables saved.")


# ------------------------------------------------------------
# 14. SAVE PROVENANCE
# ------------------------------------------------------------

provenance = {

    "project": "SurfaceTarget-CRC",

    "patch": "3.1",

    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "source": "Human Protein Atlas",

    "source_urls": {
        name: source["url"]
        for name, source in HPA_SOURCES.items()
    },

    "source_sha256": {
        record["dataset"]: record["sha256"]
        for record in source_records
    },

    "identifier_matching": (
        "Version-stripped Ensembl gene IDs"
    ),

    "transcriptomic_screening_genes":
        int(len(integrated)),

    "cross_omics_supported_genes":
        int(integrated["protein_enriched"].sum()),

    "subcellular_annotated_candidates":
        int(
            integrated[
                "hpa_subcellular_record_available"
            ].sum()
        ),

    "normal_ihc_annotated_candidates":
        int(
            integrated[
                "hpa_normal_ihc_record_available"
            ].sum()
        ),

    "plasma_membrane_annotation_candidates":
        int(
            integrated[
                "hpa_plasma_membrane_reported"
            ].fillna(False).sum()
        ),

    "final_target_ranking_performed": False,

    "extracellular_topology_verified": False,

    "normal_tissue_safety_established": False,

    "candidate_exclusions": 0,

    "limitations": [
        "HPA IF annotations derive from examined cell models.",
        "Absence of an IF annotation does not establish absence of a protein.",
        "Plasma-membrane localization does not prove extracellular epitope accessibility.",
        "IHC staining is categorical and depends on antibody performance.",
        "Normal-tissue expression does not by itself establish clinical toxicity.",
        "RNA and proteomic measurement limitations remain applicable.",
        "HPA evidence requires integration with UniProt topology and other resources.",
        "No therapeutic targets have been validated."
    ]
}

with open(
    META / "hpa_annotation_audit_provenance.json",
    "w"
) as output:

    json.dump(
        provenance,
        output,
        indent=4
    )

print("\nResearch provenance saved.")


# ------------------------------------------------------------
# 15. FINAL SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("PATCH 3.1 SUMMARY")
print("=" * 65)

print("\nTranscriptomic screening genes:", len(integrated))

print(
    "Cross-omics-supported genes:",
    int(integrated["protein_enriched"].sum())
)

print(
    "\nHPA subcellular annotations:",
    int(
        integrated[
            "hpa_subcellular_record_available"
        ].sum()
    )
)

print(
    "HPA normal-tissue IHC annotations:",
    int(
        integrated[
            "hpa_normal_ihc_record_available"
        ].sum()
    )
)

print(
    "Reported plasma-membrane annotations:",
    int(
        integrated[
            "hpa_plasma_membrane_reported"
        ].fillna(False).sum()
    )
)

print("\nSource datasets downloaded or reused: 2")

print("Annotation and audit tables saved: 6")

print("Source manifest and provenance saved.")

print("\nNo candidate genes excluded.")

print("No therapeutic target scores assigned.")

print("No TCGA or CPTAC results overwritten.")

print("\nPATCH 3.1 COMPLETE")

print("=" * 65)